# CESLR — Continuous Ethiopian Sign Language Recognition (Kaggle training)

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**, Internet **ON**
2. Attach your dataset (Add Input). It must contain, under one root:
   - `features/fullFrame-256x256px/{train,dev,test}/<clip_id>/1/*.png` — frames already resized to 256×256
   - `annotations/manual/{train,dev,test}.corpus.csv` — the gloss annotations

   Resize locally first with:
   `cd preprocess && python dataset_preprocess.py --process-image --multiprocessing`

**Annotations are regenerated here from your CSVs** (cell 3), so the gloss dict, info
files and ground-truth STMs always match the dataset you actually attached — however many
sentences it has. Nothing is assumed from the repo.

Times below are for a ~1k-clip dataset on one T4 (~2 s/step at batch 2). Scale by your
clip count: `steps/epoch = clips / batch_size`. Checkpoints save every epoch and resume
automatically, so a run can span several 12-hour sessions.

In [ ]:
import os

BRANCH = 'fix/macos-eval-and-tooling'  # set to 'main' once this is merged

if not os.path.exists('/kaggle/working/CESLR'):
    !git clone -b {BRANCH} https://github.com/ethio-artifical/CESLR.git /kaggle/working/CESLR
%cd /kaggle/working/CESLR
!pip -q install pyctcdecode
!git log --oneline -1

In [ ]:
# Link the attached dataset (frames + annotation CSVs) into the paths the repo expects.
# /kaggle/input is read-only, so we symlink rather than copy.
import glob, os

frames_src = glob.glob('/kaggle/input/**/fullFrame-256x256px', recursive=True)
assert frames_src, ('No fullFrame-256x256px folder under /kaggle/input. Attach a dataset '
                    'whose frames are already resized to 256x256.')
anno_src = glob.glob('/kaggle/input/**/annotations/manual', recursive=True)
assert anno_src, ('No annotations/manual folder under /kaggle/input. The dataset must '
                  'include {train,dev,test}.corpus.csv.')

root = 'dataset/CESLR/CESLR-multisigner'
os.makedirs(f'{root}/features', exist_ok=True)
os.makedirs(f'{root}/annotations', exist_ok=True)
for src, dst in [(frames_src[0], f'{root}/features/fullFrame-256x256px'),
                 (anno_src[0], f'{root}/annotations/manual')]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f'{dst} -> {os.path.realpath(dst)}')

for split in ['train', 'dev', 'test']:
    clips = os.listdir(f'{root}/features/fullFrame-256x256px/{split}')
    csv = f'{root}/annotations/manual/{split}.corpus.csv'
    rows = sum(1 for _ in open(csv, encoding='utf-8')) - 1  # minus header
    print(f'{split:5} {len(clips):5} clip folders, {rows:5} annotation rows')

In [ ]:
# Regenerate annotations from the attached CSVs, then check everything lines up.
# --input-res 256x256px points frame counting at the frames we actually have; no
# --process-image, since they are already resized (and /kaggle/input is read-only).
!cd preprocess && python dataset_preprocess.py --input-res 256x256px
!python preprocess/fix_annotations.py

import numpy as np, cv2, glob, os

assert 'import ctcdecode' not in open('utils/decode.py').read(), \
    'utils/decode.py still imports linux-only ctcdecode — wrong branch?'

gd = np.load('preprocess/CESLR/gloss_dict.npy', allow_pickle=True).item()
num_classes = len(gd) + 1
print(f'\nglosses: {len(gd)} -> num_classes: {num_classes} (blank included)')
assert sorted(v[0] for v in gd.values()) == list(range(1, len(gd) + 1)), \
    'gloss indices are not contiguous from 1'

ok = True
for split in ['train', 'dev', 'test']:
    info = np.load(f'preprocess/CESLR/{split}_info.npy', allow_pickle=True).item()
    ids = [v['fileid'] for v in info.values() if isinstance(v, dict)]
    oov = {w for v in info.values() if isinstance(v, dict)
           for w in v['label'].split() if w not in gd}
    zero = [v['fileid'] for v in info.values()
            if isinstance(v, dict) and v['num_frames'] == 0]
    # The evaluator reads the STMs in evaluation/slr_eval, not preprocess/CESLR.
    stm = [l.split()[0] for l in
           open(f'evaluation/slr_eval/CESLR-groundtruth-{split}.stm', encoding='utf-8')
           if l.strip()]
    match = set(stm) == set(ids)
    ok &= match and not oov and not zero
    print(f'{split:5} clips={len(ids):5} stm={len(stm):5} ids_match={match} '
          f'oov={oov or "none"} zero_frame_clips={len(zero)}')

sample = np.load('preprocess/CESLR/train_info.npy', allow_pickle=True).item()[0]
frames = glob.glob(f"dataset/CESLR/CESLR-multisigner/features/fullFrame-256x256px/{sample['folder']}")
assert frames, f"no frames for {sample['fileid']} — check the dataset layout"
h, w = cv2.imread(frames[0]).shape[:2]
print(f"\nsample {sample['fileid']}: {len(frames)} frames at {w}x{h}")
assert (w, h) == (256, 256), f'frames are {w}x{h}, not 256x256 — resize them first'
assert ok, 'annotation/STM mismatch above — do not train on this'
print('\nall checks passed')

In [ ]:
# Train. Auto-resumes from the newest checkpoint it can find.
#
# Cross-session resume: /kaggle/working is wiped between sessions. To continue a
# previous run, attach that notebook version's Output as an extra input — the
# checkpoints in it are picked up below automatically.
#
# If you hit CUDA OOM: the longest clips dominate memory. Drop batch_size to 1 in
# configs/baseline.yaml, or use --device 0 (single GPU) which halves the per-step
# batch the backbone sees.
import glob, os, re

EPOCHS = 40   # baseline.yaml ships 70; 40 is usually enough. None = use the config.
DEVICE = '0,1'

ckpts = sorted(glob.glob('work_dir/baseline_res18/*.pt')
               + glob.glob('/kaggle/input/**/dev_*_model.pt', recursive=True),
               key=os.path.getmtime)
resume = f'--load-checkpoints {ckpts[-1]}' if ckpts else ''
epochs = f'--num-epoch {EPOCHS}' if EPOCHS else ''
print('resuming from:', ckpts[-1] if ckpts else 'scratch')

!python main.py --config configs/baseline.yaml --device {DEVICE} {epochs} {resume}

In [ ]:
# Evaluate the best dev checkpoint on dev + test, then show example predictions.
import glob, re

ckpts = glob.glob('work_dir/baseline_res18/dev_*_model.pt')
assert ckpts, 'no dev checkpoint found yet — run the training cell first'
best = min(ckpts, key=lambda p: float(re.search(r'dev_([\d.]+)_', p).group(1)))
print('best checkpoint:', best, '\n')

!python main.py --config configs/baseline.yaml --device 0 --phase test --load-weights {best}
!python show_predictions.py --work-dir ./work_dir/baseline_res18/ --mode test